# Event Hub Producer — Wikipedia Recent Changes

Streams live edit events from the Wikimedia `recentchange` SSE feed and forwards
them to the personal Event Hub `ivanrazumovskyi_evh`. Filtered to real edits on
English Wikipedia only, to keep the stream predictable for the consumer side.

Runs a bounded batch (fixed number of events) rather than an open-ended process.

## 1. Install dependencies

In [0]:
%pip install azure-eventhub requests-sse

In [0]:
dbutils.library.restartPython()

## 2. Configuration

In [0]:
%run ./lab3_00_config

In [0]:
wikipedia_stream_url = "https://stream.wikimedia.org/v2/stream/recentchange"
target_domain = "en.wikipedia.org"
target_change_type = "edit"

dbutils.widgets.text("n_events_to_send", "50")
n_events_to_send = int(dbutils.widgets.get("n_events_to_send"))

print("Event Hub:", eh_name)
print("Secret scope:", secret_scope)
print("Source URL:", wikipedia_stream_url)
print("Filter: type =", target_change_type, "| domain =", target_domain)
print("Events to send:", n_events_to_send)

## 3. Load Event Hub connection string from the personal secret scope

In [0]:
connection_string = dbutils.secrets.get(scope=secret_scope, key=eventhub_secret_key)
print("Connection string loaded, length:", len(connection_string))

## 4. Wikipedia recentchange generator — filtered

In [0]:
import json
from requests_sse import EventSource

def fetch_wikipedia_edits(url, user_agent, change_type, domain):
    headers = {"User-Agent": user_agent}
    with EventSource(url, headers=headers) as stream:
        for event in stream:
            if event.type != "message":
                continue
            try:
                change = json.loads(event.data)
            except ValueError:
                continue

            # Skip Wikimedia's synthetic health-check events
            if change.get("meta", {}).get("domain") == "canary":
                continue

            # Keep only real edits on the target project
            if change.get("type") != change_type:
                continue
            if change.get("meta", {}).get("domain") != domain:
                continue

            yield change

## 5. Send filtered events to Event Hub

In [0]:
import time
from datetime import datetime
from azure.eventhub import EventHubProducerClient, EventData

def run_producer(connection_string, n_events):
    producer = EventHubProducerClient.from_connection_string(conn_str=connection_string)
    sent_events = 0

    try:
        with producer:
            for change in fetch_wikipedia_edits(
                wikipedia_stream_url, login, target_change_type, target_domain
            ):
                batch = producer.create_batch()
                batch.add(EventData(json.dumps(change, ensure_ascii=False)))
                producer.send_batch(batch)
                sent_events += 1

                if sent_events % 25 == 0 or sent_events == 1:
                    print(f"[{datetime.now():%H:%M:%S}] "
                          f"Sent {sent_events}/{n_events}: "
                          f"{change.get('user')} edited '{change.get('title')}'")

                if sent_events >= n_events:
                    break
    except Exception as e:
        print(f"Producer stopped with error after {sent_events} events: {e}")
        raise

    return sent_events

total_sent = run_producer(connection_string, n_events_to_send)
print(f"\n✅ Producer finished. Total events sent: {total_sent}")